# Replication Walkthrough: He, Kelly & Manela (2017)
## *Intermediary Asset Pricing: New Evidence from Many Asset Classes*

**Journal of Financial Economics, 126(1), 2017**

---

This notebook walks through the replication pipeline for the core empirical outputs of He, Kelly & Manela (HKM) 2017. We focus on:

1. **Table 2** — Primary dealer (PD) size relative to comparison groups (broker-dealers, banks, all public firms)
2. **Table 3** — Summary statistics: intermediary capital ratios and macroeconomic variables
3. **Figure 1** — Intermediary capital ratio and capital risk factor (standardized)
4. **Figure 4** — Comparison of three intermediary capital measures (market-based, book-based, AEM)

### Paper Overview

The central hypothesis is that **shocks to the equity capital ratio of NY Fed Primary Dealer holding companies** serve as a priced risk factor across many asset classes. Primary dealers are the ~20 financial institutions authorized to transact directly with the Federal Reserve, placing them at the center of financial intermediation.

The **intermediary capital ratio** is defined as:
$$\eta_t = \frac{\sum_i \text{MarketEquity}_{i,t}}{\sum_i (\text{BookDebt}_{i,t} + \text{MarketEquity}_{i,t})}$$

The **capital risk factor** is the AR(1) innovation in $\eta_t$, scaled by lagged $\eta_t$.

---

## Setup

All project configuration (data paths, WRDS credentials, sample dates) is managed via `config.py`, which reads from a `.env` file. The key settings are:

- `START_DATE = '1960-01-01'` — Compustat data pull start
- `END_DATE = '2012-12-31'` — original paper sample end
- `UPDATED_END_DATE = '2020-12-31'` — extended replication sample end
- `DATA_DIR` — where parquet files are cached
- `OUTPUT_DIR` — where tables and figures are saved

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import sys
sys.path.insert(0, 'src')

import config

print(f"START_DATE:        {config.START_DATE}")
print(f"END_DATE:          {config.END_DATE}")
print(f"UPDATED_END_DATE:  {config.UPDATED_END_DATE}")
print(f"DATA_DIR:          {config.DATA_DIR}")
print(f"OUTPUT_DIR:        {config.OUTPUT_DIR}")

START_DATE:        1960-01-01
END_DATE:          2012-12-31
UPDATED_END_DATE:  2020-12-31
DATA_DIR:          /Users/suniltrivedi/Documents/UChicago/Academic/Winter/Full_Stack/Final_Project/p07_he_kelly_manela_2017/_data
OUTPUT_DIR:        /Users/suniltrivedi/Documents/UChicago/Academic/Winter/Full_Stack/Final_Project/p07_he_kelly_manela_2017/_output


---
## Part 1: Table 2 — Primary Dealer Size Relative to Comparison Groups

### 1.1 What Table 2 Measures

Table 2 answers: *How large are primary dealers relative to the broader financial sector?* It reports, for each of four balance sheet items (total assets, book debt, book equity, market equity), the ratio:

$$\text{Ratio} = \frac{\text{PD total}}{\text{PD total} + \text{Comparison group total}}$$

Comparison groups are:
- **BD** — other (non-PD) broker-dealers (SIC 6211)
- **Banks** — commercial banks (SIC 6020/6021/6022)
- **Cmpust.** — all US public firms in CRSP-Compustat

The three time periods reported are: full sample (1960–2012), early (1960–1990), and late (1990–2012).

### 1.2 Data Pipeline: Pulling from WRDS

The data pull is handled by `pull_table02_data.py`, which is run once via `doit` and saves results as parquet files. This separation of *pulling* from *analysis* is a core reproducible-pipeline practice — raw data is cached so analysis can be iterated quickly without re-querying WRDS.

The key steps in the pull are:

In [2]:
# This cell illustrates the data pull logic (does not re-run the pull).
# The actual pull is triggered via: doit pull_table02_data
# Results are cached to config.DATA_DIR / 'pulled/'

# Step 1: Build the CCM (CRSP-Compustat Merged) gvkey universe
# Restricted to US-incorporated firms (fic='USA') to exclude foreign ADRs
# whose full consolidated balance sheets would inflate the Cmpust. denominator.
CCM_QUERY_SKETCH = """
SELECT DISTINCT l.gvkey
FROM crsp.ccmxpf_lnkhist l
JOIN comp.company c ON l.gvkey = c.gvkey
WHERE l.linktype LIKE 'L%'
  AND (l.linkprim = 'C' OR l.linkprim = 'P')
  AND c.fic = 'USA'        -- domestic firms only
  AND l.linkdt <= '{end_date}'
  AND (l.linkenddt IS NULL OR l.linkenddt >= '{start_date}')
"""

# Step 2: Build BD and Banks gvkey universes via Compustat SIC codes
# Critically uses BOTH indfmt='INDL' and indfmt='FS' — bank holding
# companies (JPMorgan, Citigroup, BofA) file under 'FS' format and
# would be silently omitted from the denominator with INDL-only filters.
BD_QUERY_SKETCH = """
SELECT DISTINCT f.gvkey
FROM comp.funda f JOIN comp.company c ON f.gvkey = c.gvkey
WHERE (c.sic IN ('6211') OR f.sich IN ('6211'))  -- broker-dealers
  AND c.fic = 'USA'
  AND (f.indfmt = 'INDL' OR f.indfmt = 'FS')   -- both formats!
  AND f.datafmt = 'STD' AND f.popsrc = 'D' AND f.consol = 'C'
"""

print("Data pull targets (saved to _data/pulled/):")
pull_targets = [
    "table02_ccm_gvkeys.parquet   — universe of CCM-linked US firms",
    "table02_bd_gvkeys.parquet    — broker-dealer (SIC 6211) gvkeys",
    "table02_banks_gvkeys.parquet — commercial bank (SIC 6020/21/22) gvkeys",
    "table02_raw_PD.parquet       — quarterly fundamentals for primary dealers",
    "table02_raw_BD.parquet       — quarterly fundamentals for BD comparison group",
    "table02_raw_Banks.parquet    — quarterly fundamentals for Banks comparison group",
    "table02_raw_Cmpust.parquet   — quarterly fundamentals for all-Compustat group",
]
for t in pull_targets:
    print(f"  {t}")

Data pull targets (saved to _data/pulled/):
  table02_ccm_gvkeys.parquet   — universe of CCM-linked US firms
  table02_bd_gvkeys.parquet    — broker-dealer (SIC 6211) gvkeys
  table02_banks_gvkeys.parquet — commercial bank (SIC 6020/21/22) gvkeys
  table02_raw_PD.parquet       — quarterly fundamentals for primary dealers
  table02_raw_BD.parquet       — quarterly fundamentals for BD comparison group
  table02_raw_Banks.parquet    — quarterly fundamentals for Banks comparison group
  table02_raw_Cmpust.parquet   — quarterly fundamentals for all-Compustat group


### 1.3 Loading the Pre-Pulled Data

Once the data pull has run, we load directly from parquet. This is the same approach used in `Table02Prep.main()`.

In [3]:
import Table02Prep

PULLED_DIR = config.DATA_DIR / 'pulled'

# Load pre-pulled WRDS data from parquet
ds = {
    'PD':      pd.read_parquet(PULLED_DIR / 'table02_raw_PD.parquet'),
    'BD':      pd.read_parquet(PULLED_DIR / 'table02_raw_BD.parquet'),
    'Banks':   pd.read_parquet(PULLED_DIR / 'table02_raw_Banks.parquet'),
    'Cmpust.': pd.read_parquet(PULLED_DIR / 'table02_raw_Cmpust.parquet'),
}

for name, df in ds.items():
    print(f"{name:8s}: {len(df):>7,} firm-quarters, "
          f"{df['gvkey'].nunique():>4} unique gvkeys, "
          f"dates {pd.to_datetime(df['datadate']).min().date()} "
          f"to {pd.to_datetime(df['datadate']).max().date()}")

PD      :   3,001 firm-quarters,   45 unique gvkeys, dates 1961-12-31 to 2020-12-31
BD      :   9,359 firm-quarters,  157 unique gvkeys, dates 1962-03-31 to 2020-12-31
Banks   :  79,495 firm-quarters, 1228 unique gvkeys, dates 1961-12-31 to 2020-12-31
Cmpust. : 1,364,688 firm-quarters, 25972 unique gvkeys, dates 1961-03-31 to 2020-12-31


### 1.4 Key Implementation Details

Several non-obvious choices in the pipeline are worth highlighting:

**Exclusion of non-financial conglomerates from PD numerator.** GE (gvkey 005047) and Sears (006307) held Kidder Peabody and Dean Witter respectively, making them technically registered primary dealers. But their full consolidated balance sheets — appliances, retail stores — are not representative of broker-dealer capital. They are excluded.

In [4]:
# Non-financial PD gvkeys excluded from the numerator
print("Excluded from PD numerator:")
print("  gvkey 005047 = GE (held Kidder Peabody 1986-1994)")
print("  gvkey 006307 = Sears (held Dean Witter Reynolds 1977-1993)")
print()
print("These are identified in Table02Prep.NON_FINANCIAL_PD_GVKEYS:")
print(" ", Table02Prep.NON_FINANCIAL_PD_GVKEYS)

Excluded from PD numerator:
  gvkey 005047 = GE (held Kidder Peabody 1986-1994)
  gvkey 006307 = Sears (held Dean Witter Reynolds 1977-1993)

These are identified in Table02Prep.NON_FINANCIAL_PD_GVKEYS:
  {'006307', '005047'}


**Time-varying PD exclusion from Banks denominator.** A bank that *later* becomes a primary dealer should still count toward the Banks comparison group *before* its PD stint begins. For example, Citigroup (gvkey 3066) was a commercial bank before becoming a primary dealer. A static exclusion would remove all of Citicorp's ~\$1.9T of assets from the Banks denominator for all years — a large error. Instead, we build a month-by-month schedule and only zero out a gvkey's contribution during its active PD months.

In [5]:
# Load the primary dealer link table
merged_main = Table02Prep.clean_primary_dealers_data(
    fname='Primary_Dealer_Link_Table3_DOMESTIC.csv'
)

print(f"Primary dealers in link table: {merged_main['Primary Dealer'].nunique()} unique names")
print(f"Unique gvkeys: {merged_main['gvkey'].nunique()}")
print()
print("Sample rows:")
merged_main[['Primary Dealer', 'gvkey', 'Start Date', 'End Date']].head(8)

                   Primary Dealer  Start Date    End Date  gvkey
0               BA SECURITIES INC  04/18/1994  09/30/1997   2024
1  BANC OF AMERICA SECURITIES LLC  05/17/1999  11/01/2010   7647
2    BANC ONE CAPITAL MARKETS INC  04/01/1999  08/01/2004   1998
3   BANCAMERICA ROBERTSON STEPHEN  10/01/1997  08/31/1998   2024
4      BANCAMERICA SECURITIES INC  09/01/1998  09/30/1998   2024
Unique dealers: 76
Missing start dates: 0.0
Missing end dates: 0.0
Missing gvkey: 0.0
Primary dealers in link table: 76 unique names
Unique gvkeys: 47

Sample rows:


,Primary Dealer,gvkey,Start Date,End Date
0,BA SECURITIES INC,2024,04/18/1994,09/30/1997
1,BANC OF AMERICA SECURITIES LLC,7647,05/17/1999,11/01/2010
2,BANC ONE CAPITAL MARKETS INC,1998,04/01/1999,08/01/2004
3,BANCAMERICA ROBERTSON STEPHEN,2024,10/01/1997,08/31/1998
4,BANCAMERICA SECURITIES INC,2024,09/01/1998,09/30/1998
5,BANK OF AMERICA NT & SA,2024,11/17/1971,04/15/1994
6,BANKERS TRUST,2029,05/19/1960,07/07/1989
7,BEARSTEARNS & CO INC,11818,06/10/1981,10/01/2008


In [6]:
# Build the PD active schedule (used for time-varying Banks exclusion)
pd_active_schedule = Table02Prep.build_pd_active_schedule(
    merged_main,
    start=config.START_DATE,
    end=config.END_DATE,
)

print(f"PD active (gvkey, month) entries: {len(pd_active_schedule):,}")
print(f"Unique gvkeys with active PD periods: {pd_active_schedule['gvkey'].nunique()}")
print()
# Show which gvkeys have the most active months
top = pd_active_schedule.groupby('gvkey').size().sort_values(ascending=False).head(5)
print("Top 5 gvkeys by active PD months:")
print(top)

PD active (gvkey, month) entries: 10,927
Unique gvkeys with active PD periods: 46

Top 5 gvkeys by active PD months:
gvkey
002968    632
007267    585
007562    492
002029    469
114628    457
dtype: int64


### 1.5 Building Monthly Sector Totals

Compustat reports data quarterly, but HKM computes ratios at a monthly frequency by carrying the last available quarterly observation forward. The function `build_monthly_sector_totals_from_fundq` handles this via a firm-level resample and forward-fill, then aggregates across firms to monthly sector totals.

In [7]:
# Build month-end sector totals for all four groups
monthly_totals = Table02Prep.build_all_monthly_totals(
    ds,
    start=config.START_DATE,
    end=config.END_DATE,
    banks_pd_active_schedule=pd_active_schedule,
)

print("\nMonthly total assets ($ millions, sample mean):")
for group, df in monthly_totals.items():
    mean_ta = df['total_assets'].mean()
    print(f"  {group:8s}: ${mean_ta:>15,.0f} M")


[PD] MONTHLY TOTALS DIAGNOSTICS
  rows / expected: 636 / 636
  total_assets non-missing months: 613 (96.384%)
  missing share:
total_assets     0.036164
book_debt        0.040881
book_equity      0.040881
market_equity    0.040881
dtype: float64
  share months with ALL 4 totals present: 0.959
  total_assets mean: 2,392,464.80
  total_assets max : 10,162,236.06


[BD] MONTHLY TOTALS DIAGNOSTICS
  rows / expected: 636 / 636
  total_assets non-missing months: 494 (77.673%)
  missing share:
total_assets     0.223270
book_debt        0.223270
book_equity      0.187107
market_equity    0.135220
dtype: float64
  share months with ALL 4 totals present: 0.777
  total_assets mean: 289,965.68
  total_assets max : 1,032,885.79


[Banks] MONTHLY TOTALS DIAGNOSTICS
  rows / expected: 636 / 636
  total_assets non-missing months: 613 (96.384%)
  missing share:
total_assets     0.036164
book_debt        0.036164
book_equity      0.036164
market_equity    0.040881
dtype: float64
  share months with ALL

### 1.6 Computing Table 2 Ratios and Generating Outputs

For each month, we compute PD / (PD + comparison group) for each balance sheet item. Then we take time averages over the three sample sub-periods. `Table02Analysis` handles the ratio computation, summary statistics, correlation matrix, and LaTeX export.

In [8]:
import Table02Analysis

# Compute period-average ratios (the core Table 2 output)
ratios = Table02Analysis.compute_table2_ratios(
    monthly_totals,
    start=config.START_DATE,
    end=config.END_DATE,
)

print("Ratio DataFrame shape:", ratios.shape)
print("Columns:", ratios.columns.tolist()[:6], "...")
print()
print("Sample (first 6 months, total assets ratios):")
cols_to_show = ['mdate', 'total_assets_BD', 'total_assets_Banks', 'total_assets_Cmpust.']
if all(c in ratios.columns for c in cols_to_show):
    print(ratios[cols_to_show].head(6).to_string(index=False))


[RATIO DIAGNOSTICS] Group=BD
  aligned months: 636
  PD total_assets missing: 3.616%
  BD total_assets missing: 22.327%
  total_assets: denom==0 0.000%, denom NA 3.616%, ratio NA 3.616%, ratio>1 0.000%, ratio<0 0.000%
    top3: [('1961-12-31', 1.0), ('1966-12-31', 1.0), ('1969-05-31', 1.0)]
    bot3: [('1992-12-31', 0.7864), ('1993-02-28', 0.7868), ('1993-01-31', 0.787)]
  book_debt: denom==0 0.000%, denom NA 4.088%, ratio NA 4.088%, ratio>1 0.000%, ratio<0 0.000%
    top3: [('1962-03-31', 1.0), ('1968-05-31', 1.0), ('1969-06-30', 1.0)]
    bot3: [('1992-12-31', 0.7803), ('1993-02-28', 0.7807), ('1993-01-31', 0.7809)]
  book_equity: denom==0 0.000%, denom NA 4.088%, ratio NA 4.088%, ratio>1 0.000%, ratio<0 0.000%
    top3: [('1962-03-31', 1.0), ('1962-04-30', 1.0), ('1967-11-30', 1.0)]
    bot3: [('1998-10-31', 0.8168), ('1998-11-30', 0.8233), ('2000-10-31', 0.8271)]
  market_equity: denom==0 0.000%, denom NA 4.088%, ratio NA 4.088%, ratio>1 0.000%, ratio<0 0.000%
    top3: [('1962-03

In [9]:
# Generate the final Table 2 — period averages across the three sub-periods
final_table2 = Table02Analysis.summarize_table2(ratios, UPDATED=False)

print("\n=== TABLE 2: PD Share of Total (Original Sample) ===")
print(final_table2.round(3).to_string())
print()
print("Interpretation: Values near 1.0 mean PDs dominate; near 0.5 means PDs are")
print("roughly equal in size to the comparison group.")


[PERIOD DIAGNOSTICS] 1960-2012
  months in period: 636
  share months complete across ALL columns: 0.959
  total_assets means: {'total_assets_BD': 0.9324448721582638, 'total_assets_Banks': 0.4746344264501102, 'total_assets_Cmpust.': 0.24728696439834527}

[PERIOD DIAGNOSTICS] 1960-1990
  months in period: 372
  share months complete across ALL columns: 0.930
  total_assets means: {'total_assets_BD': 0.9679154310442482, 'total_assets_Banks': 0.4201077146743728, 'total_assets_Cmpust.': 0.2848813179629234}

[PERIOD DIAGNOSTICS] 1990-2012
  months in period: 276
  share months complete across ALL columns: 1.000
  total_assets means: {'total_assets_BD': 0.8848917321897325, 'total_assets_Banks': 0.5428045675485804, 'total_assets_Cmpust.': 0.19544987102926614}

=== TABLE 2: PD Share of Total (Original Sample) ===
Metric    Total assets                Book debt                Book equity                Market equity               
Source              BD  Banks Cmpust.        BD  Banks Cmpust. 

In [10]:
# Generate companion outputs: summary statistics table, figure, and correlation matrix
# (These are also exported to _output/ as LaTeX and PNG files)
Table02Analysis.create_summary_stat_table_for_data(ds, UPDATED=False)
Table02Analysis.create_figure_for_data(ratios, UPDATED=False)
Table02Analysis.create_corr_matrix_for_data(ds, UPDATED=False)

# Export the main table as LaTeX
Table02Prep.convert_and_export_table_to_latex(final_table2, UPDATED=False)

print("\nAll Table 2 outputs saved to:", config.OUTPUT_DIR)

Summary stats LaTeX saved to: /Users/suniltrivedi/Documents/UChicago/Academic/Winter/Full_Stack/Final_Project/p07_he_kelly_manela_2017/_output/table02_sstable.tex
Figure saved to: /Users/suniltrivedi/Documents/UChicago/Academic/Winter/Full_Stack/Final_Project/p07_he_kelly_manela_2017/_output/table02_figure.png
Correlation matrix LaTeX saved to: /Users/suniltrivedi/Documents/UChicago/Academic/Winter/Full_Stack/Final_Project/p07_he_kelly_manela_2017/_output/table02_corr.tex
Table 02 LaTeX saved to: /Users/suniltrivedi/Documents/UChicago/Academic/Winter/Full_Stack/Final_Project/p07_he_kelly_manela_2017/_output/table02_fixed.tex

All Table 2 outputs saved to: /Users/suniltrivedi/Documents/UChicago/Academic/Winter/Full_Stack/Final_Project/p07_he_kelly_manela_2017/_output


### 1.7 Updated Table 2 (Extended Sample)

The same pipeline runs for the updated (post-2012) sample by passing `UPDATED=True`. The analysis windows widen to `config.UPDATED_END_DATE` while the underlying parquet data is shared.

In [11]:
# Build updated-sample PD active schedule
pd_active_schedule_upd = Table02Prep.build_pd_active_schedule(
    merged_main,
    start=config.START_DATE,
    end=config.UPDATED_END_DATE,
)

monthly_totals_upd = Table02Prep.build_all_monthly_totals(
    ds,
    start=config.START_DATE,
    end=config.UPDATED_END_DATE,
    banks_pd_active_schedule=pd_active_schedule_upd,
)

ratios_upd = Table02Analysis.compute_table2_ratios(
    monthly_totals_upd,
    start=config.START_DATE,
    end=config.UPDATED_END_DATE,
)

final_table2_upd = Table02Analysis.summarize_table2(ratios_upd, UPDATED=True)

print("=== TABLE 2: PD Share of Total (Updated Sample) ===")
print(final_table2_upd.round(3).to_string())

Table02Analysis.create_figure_for_data(ratios_upd, UPDATED=True)
Table02Analysis.create_corr_matrix_for_data(ds, UPDATED=True)
Table02Prep.convert_and_export_table_to_latex(final_table2_upd, UPDATED=True)


[PD] MONTHLY TOTALS DIAGNOSTICS
  rows / expected: 732 / 732
  total_assets non-missing months: 709 (96.858%)
  missing share:
total_assets     0.031421
book_debt        0.035519
book_equity      0.035519
market_equity    0.035519
dtype: float64
  share months with ALL 4 totals present: 0.964
  total_assets mean: 3,386,537.53
  total_assets max : 12,699,841.00


[BD] MONTHLY TOTALS DIAGNOSTICS
  rows / expected: 732 / 732
  total_assets non-missing months: 590 (80.601%)
  missing share:
total_assets     0.193989
book_debt        0.193989
book_equity      0.162568
market_equity    0.117486
dtype: float64
  share months with ALL 4 totals present: 0.806
  total_assets mean: 350,389.34
  total_assets max : 1,032,885.79


[Banks] MONTHLY TOTALS DIAGNOSTICS
  rows / expected: 732 / 732
  total_assets non-missing months: 709 (96.858%)
  missing share:
total_assets     0.031421
book_debt        0.031421
book_equity      0.031421
market_equity    0.035519
dtype: float64
  share months with ALL

---
## Part 2: Table 3 — Summary Statistics for Capital Ratios and Macro Variables

### 2.1 What Table 3 Measures

Table 3 provides the descriptive statistics for the key time series used throughout the paper. **Panel A** covers *levels* of capital ratios and macroeconomic variables. **Panel B** covers *factors* (innovations/growth rates). For each series the table reports: mean, standard deviation, and pairwise correlations.

The three intermediary capital measures are:

| Measure | Definition |
|---|---|
| **Market capital ratio** | $\sum$ MarketEquity / $\sum$ (BookDebt + MarketEquity) for PD holding companies |
| **Book capital ratio** | $\sum$ BookEquity / $\sum$ (BookDebt + BookEquity) for PD holding companies |
| **AEM leverage** | BD Financial Assets / (BD Financial Assets − BD Liabilities), from Federal Reserve Z.1 |

AEM refers to Adrian, Etula & Muir (2014), who proposed the Z.1 broker-dealer leverage measure.

### 2.2 Loading the Primary Dealer Data (Table 3 Pipeline)

Unlike Table 2, which uses all firms in each comparison group, Table 3 focuses on the **primary dealer holding companies** themselves — the numerator of the capital ratio. The data is fetched from Compustat via WRDS using the hand-curated `Primary_Dealer_Link_Table3_DOMESTIC.csv` file, then merged with Z.1 broker-dealer data from FRED.

In [12]:
import Table03
import Table03Load

# The Table 3 pipeline requires a WRDS connection to fetch PD data.
# The fetch_data_for_tickers() call runs firm-by-firm over the PD link table.
# In the automated pipeline (via doit), this data is already available from
# the pull_table02_data step. Here we show the structure of the loaded data.

# If you have a live WRDS connection, uncomment the block below:
# import wrds
# prim_dealers = Table02Prep.clean_primary_dealers_data('Primary_Dealer_Link_Table3_DOMESTIC.csv')
# db = wrds.Connection(wrds_username=config.WRDS_USERNAME)
# dataset, empty_tickers = Table03Load.fetch_data_for_tickers(prim_dealers, db)
# db.close()

print("Compustat fields pulled for each PD holding company (quarterly):")
fields = {
    'datafqtr':     'Fiscal quarter (e.g. 2005Q3)',
    'atq':          'Total assets → total_assets',
    'atq - ceqq':   'Book debt → book_debt',
    'ceqq':         'Common equity → book_equity',
    'cshoq * prccq':'Shares outstanding × price → market_equity',
}
for field, desc in fields.items():
    print(f"  {field:20s}  {desc}")

Compustat fields pulled for each PD holding company (quarterly):
  datafqtr              Fiscal quarter (e.g. 2005Q3)
  atq                   Total assets → total_assets
  atq - ceqq            Book debt → book_debt
  ceqq                  Common equity → book_equity
  cshoq * prccq         Shares outstanding × price → market_equity


### 2.3 The Z.1 Broker-Dealer Data (AEM Leverage)

The AEM leverage measure comes from the Federal Reserve's Financial Accounts (Z.1 release). Two series are used:
- `BOGZ1FL664090005Q` — Security brokers & dealers: financial assets
- `BOGZ1FL664190005Q` — Security brokers & dealers: liabilities

These are pulled via `pandas_datareader` from FRED and cached locally. The leverage ratio is:
$$\text{AEM Leverage} = \frac{\text{Financial Assets}}{\text{Financial Assets} - \text{Liabilities}} = \frac{1}{\text{Equity / Assets}}$$

In [13]:
# Load the Z.1 broker-dealer series from FRED
bd_financials = Table03Load.load_bd_financials(end=config.END_DATE)

print("Z.1 broker-dealer data:")
print(f"  Date range: {bd_financials.index.min().date()} to {bd_financials.index.max().date()}")
print(f"  Quarters: {len(bd_financials)}")
print()
print(bd_financials.tail(6))

Cache not found for BD data, pulling data...
Data pulled and saved to /Users/suniltrivedi/Documents/UChicago/Academic/Winter/Full_Stack/Final_Project/p07_he_kelly_manela_2017/_data/pulled/fred_bd.parquet
Z.1 broker-dealer data:
  Date range: 1960-03-31 to 2012-12-31
  Quarters: 212

            bd_fin_assets  bd_liabilities
datafqtr                                 
2011-09-30      4277450.0       4090676.0
2011-12-31      4143391.0       3966477.0
2012-03-31      4309519.0       4123472.0
2012-06-30      4289162.0       4102860.0
2012-09-30      4359467.0       4165518.0
2012-12-31      4378225.0       4182704.0


### 2.4 Aggregating PD Data and Computing Capital Ratios

Once the firm-level PD data is assembled, we:
1. Drop duplicates and convert quarter strings (e.g. `'2005Q3'`) to dates
2. Aggregate total assets, book debt, book equity, market equity across all active PDs each quarter
3. Merge with the Z.1 BD data
4. Compute the three capital ratio measures

**Note:** Rows where `market_equity = 0` are excluded. A zero market cap in Compustat means the stock price field (`prccq`) is missing or zero — not a genuine zero-market-cap firm. Including these would drag the market capital ratio toward the book ratio.

In [14]:
# Assume `dataset` is the fetched firm-level PD data (from WRDS or cache).
# For illustration, we show what prep_dataset() does conceptually:

print("prep_dataset() pipeline steps:")
steps = [
    "1. Drop duplicate rows",
    "2. Convert 'datafqtr' strings (e.g. '2005Q3') to quarter-end dates",
    "3. Drop rows missing any of: datafqtr, total_assets, book_debt, book_equity, market_equity",
    "4. Drop rows where market_equity == 0 (missing/zero stock price in Compustat)",
    "5. Aggregate (sum) across all PD firms by quarter",
    "6. Merge with Z.1 BD financials (bd_fin_assets, bd_liabilities)",
    "7. Filter to sample window (1970Q1 to END_DATE or UPDATED_END_DATE)",
]
for s in steps:
    print(f"  {s}")

print()
print("calculate_ratios() formulas:")
print("  market_cap_ratio = market_equity / (book_debt + market_equity)")
print("  book_cap_ratio   = book_equity   / (book_debt + book_equity)")
print("  aem_leverage     = bd_fin_assets / (bd_fin_assets - bd_liabilities)")

prep_dataset() pipeline steps:
  1. Drop duplicate rows
  2. Convert 'datafqtr' strings (e.g. '2005Q3') to quarter-end dates
  3. Drop rows missing any of: datafqtr, total_assets, book_debt, book_equity, market_equity
  4. Drop rows where market_equity == 0 (missing/zero stock price in Compustat)
  5. Aggregate (sum) across all PD firms by quarter
  6. Merge with Z.1 BD financials (bd_fin_assets, bd_liabilities)
  7. Filter to sample window (1970Q1 to END_DATE or UPDATED_END_DATE)

calculate_ratios() formulas:
  market_cap_ratio = market_equity / (book_debt + market_equity)
  book_cap_ratio   = book_equity   / (book_debt + book_equity)
  aem_leverage     = bd_fin_assets / (bd_fin_assets - bd_liabilities)


In [15]:
# Run the full Table 3 pipeline for the original sample
# (requires `dataset` from the WRDS pull above)

# prep_datast    = Table03.prep_dataset(dataset, UPDATED=False)
# ratio_dataset  = Table03.aggregate_ratios(prep_datast)
# factors_dataset = Table03.convert_ratios_to_factors(ratio_dataset)

# For demonstration, load pre-computed ratio data if available:
try:
    ratio_dataset = pd.read_parquet(config.DATA_DIR / 'pulled' / 'ratio_dataset.parquet')
    factors_dataset = pd.read_parquet(config.DATA_DIR / 'pulled' / 'factors_dataset.parquet')
    print("Loaded ratio and factor datasets from cache.")
    print(f"ratio_dataset:   {len(ratio_dataset)} quarters, cols: {ratio_dataset.columns.tolist()}")
    print(f"factors_dataset: {len(factors_dataset)} quarters, cols: {factors_dataset.columns.tolist()}")
except FileNotFoundError:
    print("Ratio/factor cache not found. Run the full pipeline via: doit table03_main")
    print("Or connect to WRDS and run Table03.main() directly.")

Ratio/factor cache not found. Run the full pipeline via: doit table03_main
Or connect to WRDS and run Table03.main() directly.


### 2.5 Converting Ratios to Risk Factors

The capital **ratio** measures the *level* of intermediary health. The capital **factor** measures *shocks* to that health — the component that is unexpected given the AR(1) dynamics of the ratio. This is what prices risk in the cross-section.

For the market and book capital ratios, the factor is:
$$\eta_t^{\text{factor}} = \frac{\hat{\varepsilon}_t}{\eta_{t-1}}$$

where $\hat{\varepsilon}_t$ is the AR(1) residual. Dividing by the lagged level makes the factor scale-invariant and comparable across time.

For the AEM leverage factor, the approach differs slightly: we take the percentage change in leverage and seasonally adjust it using an additive decomposition (period=4 quarters).

In [16]:
# Illustrate the AR(1) factor construction for the market capital ratio
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.seasonal import seasonal_decompose

print("Factor construction for market_cap_ratio:")
print("""
  # 1. Fit AR(1) model on the capital ratio series
  model = AutoReg(market_cap_ratio, lags=1, trend='c')
  model_fitted = model.fit()

  # 2. Residuals are the 'unexpected' innovation
  innovations = model_fitted.resid

  # 3. Scale by lagged ratio to make factor unit-free
  market_capital_factor = innovations / market_cap_ratio.shift(1)
""")

print("Factor construction for aem_leverage:")
print("""
  # 1. Compute quarterly percentage change in AEM leverage
  leverage_growth = aem_leverage.pct_change()

  # 2. Remove seasonal component (quarterly data has annual seasonality)
  decomp = seasonal_decompose(leverage_growth, model='additive', period=4)
  aem_leverage_factor = leverage_growth - decomp.seasonal
""")

Factor construction for market_cap_ratio:

  # 1. Fit AR(1) model on the capital ratio series
  model = AutoReg(market_cap_ratio, lags=1, trend='c')
  model_fitted = model.fit()

  # 2. Residuals are the 'unexpected' innovation
  innovations = model_fitted.resid

  # 3. Scale by lagged ratio to make factor unit-free
  market_capital_factor = innovations / market_cap_ratio.shift(1)

Factor construction for aem_leverage:

  # 1. Compute quarterly percentage change in AEM leverage
  leverage_growth = aem_leverage.pct_change()

  # 2. Remove seasonal component (quarterly data has annual seasonality)
  decomp = seasonal_decompose(leverage_growth, model='additive', period=4)
  aem_leverage_factor = leverage_growth - decomp.seasonal



### 2.6 Macroeconomic Variables (Panel A)

Table 3 also reports correlations of capital ratios and factors with standard macroeconomic variables. These are loaded from multiple sources and merged:

In [17]:
print("Macroeconomic data sources:")
macro_sources = [
    ("E/P ratio",             "Shiller CAPE data (ie_data.xls from Shiller's website); E/P = 1/CAPE"),
    ("Unemployment",          "FRED: UNRATE — civilian unemployment rate"),
    ("Financial conditions",  "FRED: NFCI — Chicago Fed National Financial Conditions Index"),
    ("GDP",                   "FRED: GDPC1 — real GDP, HP-filtered (λ=1600) to remove secular trend"),
    ("Market excess return",  "Ken French data library: market factor (mkt_ret)"),
    ("Market volatility",     "CRSP value-weighted index: quarterly std dev of log daily returns"),
]
for name, source in macro_sources:
    print(f"  {name:30s}  {source}")

print()
print("Note: GDP is HP-filtered (log-level, λ=1600) to remove the secular trend.")
print("Raw log-GDP trends from ~8.3 to ~9.7 over 1970-2012, producing spuriously")
print("high correlations. The HP cyclical component is stationary and captures")
print("the business cycle, matching the paper's reported GDP correlations.")

Macroeconomic data sources:
  E/P ratio                       Shiller CAPE data (ie_data.xls from Shiller's website); E/P = 1/CAPE
  Unemployment                    FRED: UNRATE — civilian unemployment rate
  Financial conditions            FRED: NFCI — Chicago Fed National Financial Conditions Index
  GDP                             FRED: GDPC1 — real GDP, HP-filtered (λ=1600) to remove secular trend
  Market excess return            Ken French data library: market factor (mkt_ret)
  Market volatility               CRSP value-weighted index: quarterly std dev of log daily returns

Note: GDP is HP-filtered (log-level, λ=1600) to remove the secular trend.
Raw log-GDP trends from ~8.3 to ~9.7 over 1970-2012, producing spuriously
high correlations. The HP cyclical component is stationary and captures
the business cycle, matching the paper's reported GDP correlations.


In [18]:
# Load FRED macro data
macro_raw = Table03Load.load_fred_macro_data(from_cache=True)

print("FRED macro data shape:", macro_raw.shape)
print("Columns:", macro_raw.columns.tolist())
print()
print(macro_raw.tail(4))

Loaded macro data from cache.
FRED macro data shape: (3577, 3)
Columns: ['UNRATE', 'NFCI', 'GDPC1']

            UNRATE     NFCI  GDPC1
DATE                              
2026-02-06     NaN -0.55053    NaN
2026-02-13     NaN -0.54241    NaN
2026-02-20     NaN -0.53309    NaN
2026-02-27     NaN -0.52365    NaN


In [19]:
# Load Shiller CAPE data
shiller_cape = Table03Load.load_shiller_pe(from_cache=True)
shiller_ep = Table03.calculate_ep(shiller_cape)

print("Shiller E/P data:")
print(f"  Date range: {shiller_ep.index.min().date()} to {shiller_ep.index.max().date()}")
print(f"  E/P sample mean: {shiller_ep['e/p'].mean():.4f}")
print()
print(shiller_ep.tail(4))

Loading data from cache.
Shiller E/P data:
  Date range: 1871-01-31 to 2024-09-30
  E/P sample mean: 0.0680

                 e/p
date                
2024-06-30  0.028727
2024-07-31  0.028214
2024-08-31  0.028567
2024-09-30  0.028386


### 2.7 Generating Table 3 Outputs

`Table03.main()` runs all four combinations of `UPDATED` × `INCLUDE_FOREIGN` and calls `Table03Analysis` to produce Panel A, Panel B, summary statistics, and LaTeX tables. Here we run the baseline (domestic, original sample).

In [20]:
# Run Table 3 (requires WRDS connection for CRSP market volatility)
# Table03.main(UPDATED=False, INCLUDE_FOREIGN=False)

# The four variants produced:
variants = [
    ('UPDATED=False, INCLUDE_FOREIGN=False', 'table03.tex',            'Original sample, domestic PDs only'),
    ('UPDATED=True,  INCLUDE_FOREIGN=False', 'updated_table03.tex',    'Extended sample, domestic PDs only'),
    ('UPDATED=False, INCLUDE_FOREIGN=True',  'table03_intl.tex',       'Original sample, domestic + foreign PDs'),
    ('UPDATED=True,  INCLUDE_FOREIGN=True',  'updated_table03_intl.tex','Extended sample, domestic + foreign PDs'),
]

print("Table 3 output variants:")
print(f"{'Arguments':45s}  {'Output file':30s}  Description")
print("-" * 110)
for args, outfile, desc in variants:
    print(f"{args:45s}  {outfile:30s}  {desc}")

print()
print("Each variant produces:")
print("  - [prefix]table03.tex         — Panel A and B correlations")
print("  - [prefix]table03_sstable.tex — summary statistics (mean, std)")
print("  - [prefix]table03_figure03.png — time-series plot of capital ratio and macro vars")

Table 3 output variants:
Arguments                                      Output file                     Description
--------------------------------------------------------------------------------------------------------------
UPDATED=False, INCLUDE_FOREIGN=False           table03.tex                     Original sample, domestic PDs only
UPDATED=True,  INCLUDE_FOREIGN=False           updated_table03.tex             Extended sample, domestic PDs only
UPDATED=False, INCLUDE_FOREIGN=True            table03_intl.tex                Original sample, domestic + foreign PDs
UPDATED=True,  INCLUDE_FOREIGN=True            updated_table03_intl.tex        Extended sample, domestic + foreign PDs

Each variant produces:
  - [prefix]table03.tex         — Panel A and B correlations
  - [prefix]table03_sstable.tex — summary statistics (mean, std)
  - [prefix]table03_figure03.png — time-series plot of capital ratio and macro vars


---
## Part 3: Figure 1 — Intermediary Capital Ratio and Risk Factor

### 3.1 What Figure 1 Shows

Figure 1 plots the two key time series of the paper side by side, both standardized to zero mean and unit variance so they can be compared on the same scale:

- **Solid line (blue)** — The intermediary capital ratio $\eta_t$ (level)
- **Dashed line (orange)** — The capital risk factor (AR(1) innovation in $\eta_t$ divided by $\eta_{t-1}$)

NBER recessions are shaded in gray. The figure illustrates the **procyclicality** of the capital ratio — it rises in expansions and falls sharply in recessions — which is central to the paper's theory: when intermediaries are under stress (low $\eta$), they demand higher returns for holding risky assets, explaining the cross-sectional variation in expected returns.

### 3.2 Data Flow for Figure 1

In [21]:
print("Figure 1 data pipeline:")
pipeline = [
    ("1", "Table02Prep.clean_primary_dealers_data()",
     "Load PD link table (domestic dealers)"),
    ("2", "Table03Load.fetch_data_for_tickers()",
     "Pull quarterly fundamentals per PD from Compustat via WRDS"),
    ("3", "Table03Load.load_foreign_dealers()",
     "Optionally add foreign PD holding companies (Worldscope)"),
    ("4", "Table03.prep_dataset()",
     "Aggregate PD totals, merge Z.1 BD data, filter dates"),
    ("5", "Table03.aggregate_ratios()",
     "Compute market_cap_ratio, book_cap_ratio, aem_leverage"),
    ("6", "Table03.convert_ratios_to_factors()",
     "Fit AR(1) on each ratio, extract scaled innovations as factors"),
    ("7", "plot_figure01.plot_figure01()",
     "Standardize both series; overlay with NBER recession bands; save PNG"),
]
for step, fn, desc in pipeline:
    print(f"  Step {step}: {fn}")
    print(f"          → {desc}")
    print()

Figure 1 data pipeline:
  Step 1: Table02Prep.clean_primary_dealers_data()
          → Load PD link table (domestic dealers)

  Step 2: Table03Load.fetch_data_for_tickers()
          → Pull quarterly fundamentals per PD from Compustat via WRDS

  Step 3: Table03Load.load_foreign_dealers()
          → Optionally add foreign PD holding companies (Worldscope)

  Step 4: Table03.prep_dataset()
          → Aggregate PD totals, merge Z.1 BD data, filter dates

  Step 5: Table03.aggregate_ratios()
          → Compute market_cap_ratio, book_cap_ratio, aem_leverage

  Step 6: Table03.convert_ratios_to_factors()
          → Fit AR(1) on each ratio, extract scaled innovations as factors

  Step 7: plot_figure01.plot_figure01()
          → Standardize both series; overlay with NBER recession bands; save PNG



### 3.3 Plotting Logic

In [22]:
# NBER recession dates used across all figures
from datetime import datetime

NBER_RECESSIONS = [
    (datetime(1973, 11, 1), datetime(1975, 3, 1)),
    (datetime(1980, 1,  1), datetime(1980, 7, 1)),
    (datetime(1981, 7,  1), datetime(1982, 11, 1)),
    (datetime(1990, 7,  1), datetime(1991, 3, 1)),
    (datetime(2001, 3,  1), datetime(2001, 11, 1)),
    (datetime(2007, 12, 1), datetime(2009, 6, 1)),
]

def standardize(series):
    """Standardize a Series to zero mean and unit variance."""
    return (series - series.mean()) / series.std()

print("NBER recession periods shaded in Figure 1:")
for start, end in NBER_RECESSIONS:
    print(f"  {start.strftime('%b %Y')} – {end.strftime('%b %Y')}")

NBER recession periods shaded in Figure 1:
  Nov 1973 – Mar 1975
  Jan 1980 – Jul 1980
  Jul 1981 – Nov 1982
  Jul 1990 – Mar 1991
  Mar 2001 – Nov 2001
  Dec 2007 – Jun 2009


In [23]:
# Generate Figure 1 (requires ratio_dataset and factors_dataset from the pipeline above)
# plot_figure01.main(UPDATED=False)

# If ratio/factor data is available, we can run the plot directly:
import plot_figure01

try:
    plot_figure01.plot_figure01(ratio_dataset, factors_dataset, UPDATED=False)
    print("Figure 1 saved to:", config.OUTPUT_DIR / 'figure01.png')
    
    # Display inline
    from IPython.display import Image
    display(Image(str(config.OUTPUT_DIR / 'figure01.png'), width=800))
except NameError:
    print("ratio_dataset and factors_dataset not yet built.")
    print("Run the Table 3 pipeline above (requires WRDS) to generate them.")
    print("Or run from the command line: python src/plot_figure01.py")

ratio_dataset and factors_dataset not yet built.
Run the Table 3 pipeline above (requires WRDS) to generate them.
Or run from the command line: python src/plot_figure01.py


---
## Part 4: Figure 4 — Comparison of Three Intermediary Capital Measures

### 4.1 What Figure 4 Shows

Figure 4 is a two-panel comparison of the three capital measures:

- **Panel A (levels, log scale)** — Plots the market capital ratio, book capital ratio, and AEM leverage ratio over the full sample. The log scale accommodates the very different scales: capital ratios are ~5–15%, while AEM leverage is ~10–40×. High correlations between the series support the paper's claim that the market-based PD capital ratio captures the same fundamental phenomenon as other intermediary measures.

- **Panel B (risk factors, linear scale)** — Plots the AR(1) innovations for all three measures. The co-movement of shocks across measures — especially the shared 2008 trough — validates that they capture a common underlying force in intermediary balance sheet conditions.

Pairwise correlations are reported in text boxes on each panel.

### 4.2 Generating Figure 4

In [24]:
# Figure 4 uses the same ratio_dataset and factors_dataset as Figure 1,
# but plots all three measures rather than just the market capital measure.

print("Columns needed from ratio_dataset for Figure 4:")
for col, desc in [
    ('market_cap_ratio', 'PD market equity / (book debt + market equity)'),
    ('book_cap_ratio',   'PD book equity / (book debt + book equity)'),
    ('aem_leverage',     'Z.1 BD financial assets / (assets - liabilities)'),
]:
    print(f"  {col:20s}  {desc}")

print()
print("Columns needed from factors_dataset for Figure 4:")
for col, desc in [
    ('market_capital_factor', 'AR(1) innovation in market_cap_ratio, scaled by lagged ratio'),
    ('book_capital_factor',   'AR(1) innovation in book_cap_ratio, scaled by lagged ratio'),
    ('aem_leverage_factor',   'Seasonally-adjusted percentage change in aem_leverage'),
]:
    print(f"  {col:25s}  {desc}")

Columns needed from ratio_dataset for Figure 4:
  market_cap_ratio      PD market equity / (book debt + market equity)
  book_cap_ratio        PD book equity / (book debt + book equity)
  aem_leverage          Z.1 BD financial assets / (assets - liabilities)

Columns needed from factors_dataset for Figure 4:
  market_capital_factor      AR(1) innovation in market_cap_ratio, scaled by lagged ratio
  book_capital_factor        AR(1) innovation in book_cap_ratio, scaled by lagged ratio
  aem_leverage_factor        Seasonally-adjusted percentage change in aem_leverage


In [25]:
import plot_figure04

# Generate Figure 4
try:
    plot_figure04.plot_figure04(ratio_dataset, factors_dataset, UPDATED=False)
    print("Figure 4 saved to:", config.OUTPUT_DIR / 'figure04.png')

    from IPython.display import Image
    display(Image(str(config.OUTPUT_DIR / 'figure04.png'), width=850))
except NameError:
    print("ratio_dataset and factors_dataset not yet built.")
    print("Run the Table 3 pipeline above (requires WRDS) to generate them.")
    print("Or run from the command line: python src/plot_figure04.py")

ratio_dataset and factors_dataset not yet built.
Run the Table 3 pipeline above (requires WRDS) to generate them.
Or run from the command line: python src/plot_figure04.py


### 4.3 Key Design Choices in Figure 4

A few implementation details in `plot_figure04.py` are worth noting:

In [26]:
print("Panel A — log scale for levels:")
print("  Capital ratios are in percent (e.g. market_cap_ratio * 100).")
print("  AEM leverage is already in units of 'x' (e.g. 20 = 20x leverage).")
print("  Log scale allows both to be visible on the same axis despite ~10-20x")
print("  difference in magnitude.")
print("  Y-axis ticks: [5, 10, 50, 100] with ScalarFormatter (no scientific notation).")
print()
print("Panel B — pairwise correlations:")
print("  Computed via Pearson correlation after dropping NaNs on aligned series.")
print("  Displayed in text boxes with rounded corners for readability.")
print()
print("Annotation arrows:")
print("  Series are identified by annotations near their most distinctive moments:")
print("  - AEM leverage: annotated at its all-time maximum (just before 2008 crisis)")
print("  - Market capital ratio: annotated at its late-1990s tech-boom peak")
print("  - Book capital ratio: annotated at its last data point")

Panel A — log scale for levels:
  Capital ratios are in percent (e.g. market_cap_ratio * 100).
  AEM leverage is already in units of 'x' (e.g. 20 = 20x leverage).
  Log scale allows both to be visible on the same axis despite ~10-20x
  difference in magnitude.
  Y-axis ticks: [5, 10, 50, 100] with ScalarFormatter (no scientific notation).

Panel B — pairwise correlations:
  Computed via Pearson correlation after dropping NaNs on aligned series.
  Displayed in text boxes with rounded corners for readability.

Annotation arrows:
  Series are identified by annotations near their most distinctive moments:
  - AEM leverage: annotated at its all-time maximum (just before 2008 crisis)
  - Market capital ratio: annotated at its late-1990s tech-boom peak
  - Book capital ratio: annotated at its last data point


---
## Part 5: Updated Figures (Extended Sample)

The `UPDATED=True` flag extends all outputs through `config.UPDATED_END_DATE`. The underlying data is the same — only the analysis window expands. This allows readers to see how intermediary capital evolved after the original 2012Q4 sample cutoff, including the COVID-19 shock in 2020.

In [27]:
# Generate updated Figure 1 and Figure 4
try:
    # Updated Figure 1
    ratio_upd_fig, factors_upd_fig = plot_figure01._build_datasets(UPDATED=True)
    plot_figure01.plot_figure01(ratio_upd_fig, factors_upd_fig, UPDATED=True)
    print("Updated Figure 1 saved.")

    # Updated Figure 4
    plot_figure04.plot_figure04(ratio_upd_fig, factors_upd_fig, UPDATED=True)
    print("Updated Figure 4 saved.")
except Exception as e:
    print(f"Could not generate updated figures: {e}")
    print("Run from the command line: python src/plot_figure01.py --updated")
    print("                          python src/plot_figure04.py --updated")

Loading library list...
Done
                   Primary Dealer  Start Date    End Date  gvkey
0               BA SECURITIES INC  04/18/1994  09/30/1997   2024
1  BANC OF AMERICA SECURITIES LLC  05/17/1999  11/01/2010   7647
2    BANC ONE CAPITAL MARKETS INC  04/01/1999  08/01/2004   1998
3   BANCAMERICA ROBERTSON STEPHEN  10/01/1997  08/31/1998   2024
4      BANCAMERICA SECURITIES INC  09/01/1998  09/30/1998   2024
Unique dealers: 76
Missing start dates: 0.0
Missing end dates: 0.0
Missing gvkey: 0.0
No Worldscope data found for K:HSBC (1976–1994)
No Worldscope data found for NWB (1988–1989)
No Worldscope data found for J:NPCB (1992–1998)
No Worldscope data found for NWB (1984–2009)
No Worldscope data found for RBS (1984–2009)
No Worldscope data found for J:LTCR (1984–2009)
No Worldscope data found for K:HSBC (1999–2026)
No Worldscope data found for K:HSBC (1994–1999)
No Worldscope data found for MID (1975–1990)
No Worldscope data found for RBS (2009–2026)
No Worldscope data found for S

/opt/anaconda3/envs/fsfp/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QE-DEC will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/envs/fsfp/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency QE-DEC will be used.
  self._init_dates(dates, freq)


Figure 1 saved to: /Users/suniltrivedi/Documents/UChicago/Academic/Winter/Full_Stack/Final_Project/p07_he_kelly_manela_2017/_output/updated_figure01.png
Updated Figure 1 saved.
Figure 4 saved to: /Users/suniltrivedi/Documents/UChicago/Academic/Winter/Full_Stack/Final_Project/p07_he_kelly_manela_2017/_output/updated_figure04.png
Updated Figure 4 saved.


---
## Summary: Full Pipeline Map

The table below maps each `doit` task to its source module and outputs:

| doit task | Source module | Key outputs |
|---|---|---|
| `pull_table02_data` | `pull_table02_data.py` | `_data/pulled/table02_raw_*.parquet` |
| `table02_main` | `Table02Prep.py`, `Table02Analysis.py` | `table02_fixed.tex`, `table02_figure.png`, etc. |
| `pull_fred_data` | `Table03Load.py` | FRED/Shiller cache files |
| `table03_main` | `Table03.py`, `Table03Analysis.py` | `table03.tex`, `table03_sstable.tex`, etc. |
| `plot_figures` | `plot_figure01.py`, `plot_figure04.py` | `figure01.png`, `figure04.png` |

All tasks are orchestrated end-to-end by `dodo.py`. Run `doit` from the project root to execute the full pipeline.

In [28]:
import os

print("Output files generated by this replication:")
output_dir = config.OUTPUT_DIR
if output_dir.exists():
    for f in sorted(output_dir.iterdir()):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:45s}  {size_kb:>7.1f} KB")
else:
    print(f"  Output directory not yet created: {output_dir}")
    print("  Run 'doit' from the project root to generate all outputs.")

Output files generated by this replication:
  combined_document.aux                              3.8 KB
  combined_document.log                             13.3 KB
  combined_document.pdf                           2736.8 KB
  combined_document.tex                             20.4 KB
  crsp_returns_histogram.html                     4470.6 KB
  crsp_returns_timeseries.html                    4495.8 KB
  crsp_rolling_volatility.html                    4508.0 KB
  figure01.png                                     541.0 KB
  figure04.png                                     903.7 KB
  table02_corr.tex                                   2.0 KB
  table02_figure.png                               132.6 KB
  table02_fixed.tex                                  0.7 KB
  table02_sstable.tex                                1.5 KB
  table03.tex                                        1.2 KB
  table03_figure.png                               280.0 KB
  table03_figure03.png                             264.7